# RU OCR: export, ONNX, check, test of checkpoints, predictions

1. **Export**: the `CKPT` checkpoint of `CKPT_DATASET` -> Paddle inference model (`tools/export_model.py`:
   `inference.json`, `inference.pdiparams`, `inference.yml`).
2. **ONNX** for CPU users (`paddlex --paddle2onnx` in a Python 3.12 venv; only with `DEVICE = "cpu"`: an export made
   with paddlepaddle-gpu holds ops paddle2onnx does not support).
3. **Check**: `CHECK_LINES` lines through Paddle, ONNX Runtime CPU and OpenVINO CPU (PaddleX `create_model`); the
   texts must match.
   **3b. Test**: `tools/eval.py` (the training eval, every line at its own width) for every attached checkpoint
   (or those of `EVAL_CKPTS`) on every file of `EVAL_LABELS`: acc / norm_edit_dis by line length, `eval_results.tsv`.
4. **Predictions** (only with `RUN_PREDICTIONS = True`) on the lines of `LABEL_FILES`, one worker process per GPU (or
   OpenVINO on CPU), saved as they go to `preds_part{k}.tsv`, uploaded every 10 min to `PREDS_DATASET` (needs the
   `KAGGLE_API_TOKEN` secret). A session that stops is resumed by attaching `PREDS_DATASET` (or the session's output):
   lines already predicted are skipped.
5. **Disagreements**: `ru_ocr/tools/build_review.py` -> `preds_all.tsv.gz`, `review.tsv` (differ in characters),
   `review_spaces.tsv` (differ only in spaces) for the VLM notebook.

**Input**: the crop archives (`*.tarpack`) of the lines to test or predict, `ru-ocr-labels`, the checkpoint datasets.
**Settings**: `DEVICE = "cpu"` (Accelerator None, outside the GPU quota; export + ONNX + OpenVINO) or `"gpu"`
(T4 x2; faster predictions, no ONNX). Save & Run All.

**Output** (`/kaggle/working`): `model/inference`, `model/onnx`, `eval_results.tsv`; with predictions also
`preds_partial.tsv`, `preds_all.tsv.gz`, `review.tsv`, `review_spaces.tsv`.

In [ ]:
KAGGLE_USER = "vladislavershov"  # owner of the private Kaggle datasets the notebook writes to
REPO = "https://github.com/ErshovVE/PaddleOCR.git"  # cloned for the configs and ru_ocr/tools
BRANCH = "feat/ru-ppocrv6-small-training"
CKPT = "best_accuracy"   # checkpoint prefix in CKPT_DATASET: best_accuracy or latest
CKPT_DATASET = "ru-ocr-checkpoints-ft4"  # exported and checked
CONFIG = "ru_ocr/configs/ru_PP-OCRv6_small_rec.yml"
MODEL_NAME = "PP-OCRv6_small_rec"
LABEL_FILES = ["all_val5k_clean_w_h.txt", "all_val5k_w_h.txt"]  # [0] finds ru-ocr-labels, [-1] gives the check lines
DEVICE = "cpu"           # "cpu": export + ONNX, OpenVINO predictions; "gpu": Paddle predictions on 2 GPUs, no ONNX
PRED_LIMIT = None        # predict only the first lines of LABEL_FILES (None = all; e.g. 20 for a quick check)
SAVE_EVERY = 2000        # predictions appended to preds_partial.tsv every SAVE_EVERY lines (per device)
PREDS_DATASET = None     # prediction parts uploaded here every 10 min; None: no upload
# PREDS_DATASET = f"{KAGGLE_USER}/ru-ocr-stroyinf-modern-preds"  # a long prediction run, resumable
CHECK_LINES = 3          # lines for the backend check
EVAL_CKPTS = None        # datasets whose CKPT is tested; None: every attached dataset that has one
# EVAL_CKPTS = ["ru-ocr-checkpoints-ft4"]  # or only some of them, e.g. just the latest run
EVAL_LABELS = ["all_val5k_clean_w_h.txt", "all_val5k_w_h.txt"]  # files of ru-ocr-labels they are tested on
RUN_PREDICTIONS = False  # sections 4-5: predictions on LABEL_FILES and the disagreements
BATCH = 64               # prediction batch per worker
MODEL_DIR = "/kaggle/working/model"
import os, subprocess, sys, time
if DEVICE == "gpu":
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
if PREDS_DATASET:  # fail at once: without the secret nothing is uploaded and a long run leaves no trace until it ends
    from kaggle_secrets import UserSecretsClient
    try:
        UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
    except Exception as exc:
        raise RuntimeError("attach the KAGGLE_API_TOKEN secret (Add-ons -> Secrets; Kaggle drops it when a "
                           "notebook file is imported) or set PREDS_DATASET = None") from exc

## 0. Paddle GPU, PaddleX, code

In [ ]:
import shlex, signal, threading


def sh(cmd, timeout=1200, attempts=3):
    """Run a shell command with its output in the cell. Some installs stall forever with an idle CPU: a run is
    killed with its whole process group after `timeout` seconds and retried -- the same as pressing Stop by hand
    (Stop kills a hung `!` command and the cell goes on)."""
    for attempt in range(1, attempts + 1):
        print(time.strftime("%H:%M:%S"), f"[{attempt}/{attempts}]", cmd, flush=True)
        proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                start_new_session=True)
        timer = threading.Timer(timeout, os.killpg, (proc.pid, signal.SIGKILL))
        timer.start()
        for line in proc.stdout:
            print(line, end="", flush=True)
        hung = not timer.is_alive()
        timer.cancel()
        if proc.wait() == 0:
            return
        print(time.strftime("%H:%M:%S"), f"hung for {timeout} s, killed" if hung else f"exit code {proc.returncode}", flush=True)
    raise RuntimeError(f"failed {attempts} times: {cmd}")


if DEVICE == "gpu":
    paddle_pkg = "paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/"
else:
    paddle_pkg = "paddlepaddle==3.3.1"
PIP = f"{shlex.quote(sys.executable)} -m pip install --progress-bar off --timeout 120 --retries 10"
from importlib.metadata import PackageNotFoundError, version
try:
    IMAGE_NCCL = version("nvidia-nccl-cu12")
except PackageNotFoundError:
    IMAGE_NCCL = None
sh(f"{PIP} {paddle_pkg}", attempts=5)
# paddlepaddle-gpu (cu126) pins an older nvidia-nccl-cu12 over the image's one, and the image's torch (cu128) then
# fails on import (undefined symbol ncclCommShrink); paddlex imports torch through modelscope. NCCL is backward
# compatible and each predictor uses one GPU: the image's version goes back.
if DEVICE == "gpu" and IMAGE_NCCL and version("nvidia-nccl-cu12") != IMAGE_NCCL:
    sh(f"{PIP} --no-deps --force-reinstall nvidia-nccl-cu12=={IMAGE_NCCL}")
# onnxruntime-openvino: ONNX Runtime with the OpenVINO execution provider (replaces plain onnxruntime;
# the PaddleX hpi-cpu plugin, ultra-infer, has no Python 3.13 build)
sh(f"{PIP} 'paddlex[ocr-core]>=3.7,<3.8' onnxruntime-openvino")
# paddle2onnx has wheels only for Python <= 3.12 (Kaggle runs 3.13): the ONNX conversion runs in its own venv.
# ONNX serves the CPU path only (OpenVINO); a GPU export holds ops paddle2onnx does not support, so on GPU it is skipped
if DEVICE == "cpu":
    sh(f"{PIP} uv")
    sh("uv venv --clear --seed --python 3.12 /tmp/py312")  # --clear: rerunnable; --seed: pip inside, paddlex --install uses it
    sh("uv pip install --python /tmp/py312/bin/python paddlepaddle==3.3.1 'paddlex[ocr-core]>=3.7,<3.8'")
    sh("/tmp/py312/bin/paddlex --install paddle2onnx")
import onnxruntime
print("ONNX Runtime providers:", onnxruntime.get_available_providers())
# torch (imported by paddlex through modelscope) and paddle must both import after the installs
res = subprocess.run([sys.executable, "-c", "import torch, paddle; print('torch', torch.__version__, 'paddle', "
                      "paddle.__version__, 'cuda', paddle.device.is_compiled_with_cuda())"], capture_output=True, text=True)
print(res.stdout.strip(), res.stderr.strip()[-2000:] if res.returncode else "")
assert res.returncode == 0, "torch or paddle does not import, see above"

In [ ]:
%cd /kaggle/working
!rm -rf PaddleOCR && git clone -q --depth 1 --branch {BRANCH} {REPO}
%cd /kaggle/working/PaddleOCR
!git log --oneline -1
sh(f"{PIP} -r requirements.txt")

## Data, labels, weights
Crop archives `*.tarpack` are unpacked into `/tmp/ru_root`; folders of single crops (`<...>/stroyinf/<folder>` with
`crops/`) are linked there. The label folder is the one with `LABEL_FILES[0]`; earlier prediction parts
(`preds_part*.tsv`) under `/kaggle/input` are picked up for resuming.

In [ ]:
import pickle

ROOT = "/tmp/ru_root"
os.makedirs(os.path.join(ROOT, "stroyinf"), exist_ok=True)
# crops packed as *.tarpack (plain tar, see ru_ocr/tools/pack_dataset.py --format tar) are unpacked into ROOT,
# their paths already start with stroyinf/<folder>/crops/
for top, dirs, files in os.walk("/kaggle/input"):
    for name in sorted(f for f in files if f.endswith(".tarpack")):
        start = time.time()
        subprocess.run(["tar", "-xf", os.path.join(top, name), "-C", ROOT], check=True)
        print(f"unpacked {name} in {time.time() - start:.0f} s", flush=True)
labels_dir, weights, partials = None, [], []
for top, dirs, files in os.walk("/kaggle/input"):
    if "crops" in dirs:
        dirs.remove("crops")
        if os.path.basename(os.path.dirname(top)) == "stroyinf":
            link = os.path.join(ROOT, "stroyinf", os.path.basename(top))
            if not os.path.exists(link):
                os.symlink(top, link)
    if LABEL_FILES[0] in files:
        labels_dir = top
    if CKPT + ".pdparams" in files and CKPT_DATASET in top.split(os.sep):
        weights.append(os.path.join(top, CKPT))
    # preds_partial.tsv / preds_part{k}.tsv: output of an earlier session of this notebook, added as Input
    partials += [os.path.join(top, f) for f in files if f.startswith("preds_part") and f.endswith(".tsv")]
assert labels_dir and len(weights) == 1, f"attach ru-ocr-labels and {CKPT_DATASET}: {weights}"
WEIGHTS = weights[0]
states = pickle.load(open(WEIGHTS + ".states", "rb"), encoding="latin1")
print("stroyinf folders:", len(os.listdir(os.path.join(ROOT, "stroyinf"))), "labels:", labels_dir)
print("earlier predictions:", partials)
print("weights:", WEIGHTS, "global step", states.get("global_step"), "val acc", states.get("best_model_dict", {}).get("acc"))

## 1. Export: checkpoint -> inference model

In [ ]:
!python tools/export_model.py -c {CONFIG} -o Global.pretrained_model={WEIGHTS} Global.save_inference_dir={MODEL_DIR}/inference 2>&1 | grep -v -i warn | tail -3
!ls -la {MODEL_DIR}/inference

## 2. ONNX for CPU

In [ ]:
if DEVICE == "cpu":  # the GPU predictions use the Paddle model, no ONNX needed
    # PATH: paddlex looks for the paddle2onnx executable of the venv
    env = dict(os.environ, PATH="/tmp/py312/bin:" + os.environ["PATH"])
    subprocess.run(["paddlex", "--paddle2onnx", "--paddle_model_dir", f"{MODEL_DIR}/inference",
                    "--onnx_model_dir", f"{MODEL_DIR}/onnx"], env=env, check=True)
    print(sorted(os.listdir(f"{MODEL_DIR}/onnx")))

## 3. Check: Paddle (GPU, or CPU when DEVICE = "cpu"), ONNX Runtime CPU, OpenVINO CPU
The same `CHECK_LINES` lines through each backend; the texts must match.

In [ ]:
from paddlex import create_model

with open(os.path.join(labels_dir, LABEL_FILES[-1]), encoding="utf-8") as f:
    check = [line.rstrip("\n").split("\t")[:2] for _, line in zip(range(CHECK_LINES), f)]
check_paths = [os.path.join(ROOT, p) for p, _ in check]
backends = {
    f"paddle {DEVICE}": dict(model_dir=f"{MODEL_DIR}/inference", device=DEVICE + (":0" if DEVICE == "gpu" else "")),
    "onnxruntime cpu": dict(model_dir=f"{MODEL_DIR}/onnx", device="cpu", engine="onnxruntime",
                            engine_config={"providers": ["CPUExecutionProvider"]}),
    "openvino cpu": dict(model_dir=f"{MODEL_DIR}/onnx", device="cpu", engine="onnxruntime",
                         engine_config={"providers": ["OpenVINOExecutionProvider"], "provider_options": {"device_type": "CPU"}}),
}
if DEVICE == "gpu":  # no ONNX model on GPU runs
    backends = {k: v for k, v in backends.items() if k.startswith("paddle")}
texts = {}
for name, kwargs in backends.items():
    model = create_model(model_name=MODEL_NAME, **kwargs)
    start = time.time()
    res = list(model.predict(check_paths, batch_size=1))
    texts[name] = [r["rec_text"] for r in res]
    print(f"--- {name}: {time.time() - start:.2f} s")
    for (_, label), r in zip(check, res):
        print(f"  {r['rec_score']:.3f}  {r['rec_text']}  | label: {label}")
    del model
ref_name = f"paddle {DEVICE}"
for name, t in texts.items():
    print(name, f"same as {ref_name}" if t == texts[ref_name] else f"DIFFERS from {ref_name}")

## 3b. Test: `tools/eval.py` for every attached checkpoint (or those of `EVAL_CKPTS`) on every file of `EVAL_LABELS`
The training eval (same config, every line at its own width, spaces count). Metrics are read from the block after `metric eval` (the lines before it are the metrics stored in the checkpoint).

In [ ]:
import re
import pandas as pd

eval_ckpts = {}  # dataset folder name -> checkpoint prefix
for top, dirs, files in os.walk("/kaggle/input"):
    if CKPT + ".pdparams" in files:
        eval_ckpts[os.path.basename(top)] = os.path.join(top, CKPT)
if EVAL_CKPTS is not None:
    missing = set(EVAL_CKPTS) - set(eval_ckpts)
    assert not missing, f"attach {sorted(missing)}: found {sorted(eval_ckpts)}"
    eval_ckpts = {name: eval_ckpts[name] for name in EVAL_CKPTS}
assert eval_ckpts, f"no {CKPT}.pdparams under /kaggle/input: attach a checkpoint dataset"
print("tested:", sorted(eval_ckpts))
rows = []
for name, prefix in eval_ckpts.items():
    for labels in EVAL_LABELS:
        start = time.time()
        res = subprocess.run(
            [sys.executable, "tools/eval.py", "-c", CONFIG, "-o", f"Global.use_gpu={DEVICE == 'gpu'}",
             f"Global.checkpoints={prefix}", f"Eval.dataset.data_dir={ROOT}",
             f"Eval.dataset.label_file_list=[{os.path.join(labels_dir, labels)}]", "Eval.loader.num_workers=2"],
            capture_output=True, text=True)
        out = res.stdout + res.stderr
        assert res.returncode == 0 and "metric eval" in out, f"eval failed for {name} on {labels}:\n{out[-3000:]}"
        metrics = dict(re.findall(r"INFO: (\w+):([0-9.]+)", out.split("metric eval")[-1]))
        row = {"ckpt": name, "labels": labels, "minutes": round((time.time() - start) / 60, 1)}
        row.update({k: float(v) for k, v in metrics.items() if k.startswith(("acc", "norm_edit_dis", "ned_len", "n_len"))})
        rows.append(row)
        print(f"{name} on {labels}: acc {row['acc']:.4f}, ned {row['norm_edit_dis']:.4f}, {row['minutes']} min", flush=True)
results = pd.DataFrame(rows)
results.to_csv("/kaggle/working/eval_results.tsv", sep="\t", index=False)
cols = ["ckpt", "labels", "acc", "norm_edit_dis"] + [c for c in results.columns if c.startswith("acc_len")]
results[cols].round(4)

## 4. Predictions: one worker process per GPU
`pred_worker.py` runs once per GPU (`CUDA_VISIBLE_DEVICES`), each process takes every second line and appends every
`SAVE_EVERY` lines to `preds_part{k}.tsv` (separate processes: threads shared the GIL in the CPU-side preprocessing
and the GPUs worked in turns). With `DEVICE = "cpu"`: one OpenVINO worker on the ONNX model. Lines found in earlier
parts (`preds_partial.tsv`, `preds_part*.tsv` under `/kaggle/input` or in `/kaggle/working`) are skipped.
A background thread (`ru_ocr/tools/ckpt_uploader.py`) uploads the parts to `PREDS_DATASET` every 10 min and once at
the end; at the end all parts are merged into `preds_partial.tsv`.

In [ ]:
%%writefile /kaggle/working/pred_worker.py
# one predictor process: python pred_worker.py <todo tsv> <out tsv> <model kwargs json> <model name> <batch> <save every>
import json
import sys

from paddlex import create_model

todo_file, out_file, kwargs, model_name, batch, save_every = sys.argv[1:7]
batch, save_every = int(batch), int(save_every)
with open(todo_file, encoding="utf-8") as f:
    todo = [line.rstrip("\n").split("\t") for line in f]  # index, relative path, absolute path
model = create_model(model_name=model_name, **json.loads(kwargs))
for start in range(0, len(todo), save_every):
    chunk = todo[start : start + save_every]
    res = model.predict([full for _, _, full in chunk], batch_size=batch)
    lines = [f"{i}\t{rel}\t{r['rec_text']}\t{float(r['rec_score'])}\n" for (i, rel, _), r in zip(chunk, res)]
    with open(out_file, "a", encoding="utf-8") as f:
        f.writelines(lines)
print("done", len(todo), flush=True)

In [ ]:
if RUN_PREDICTIONS:  # off: export, check and test only
    import json

    rows = []
    for name in LABEL_FILES:
        with open(os.path.join(labels_dir, name), encoding="utf-8") as f:
            rows += [line.rstrip("\n").split("\t")[:2] for line in f if line.strip()]
    if PRED_LIMIT:
        rows = rows[:PRED_LIMIT]
    print(len(rows), "lines")

    PARTIAL = "/kaggle/working/preds_partial.tsv"
    preds = {}
    for path in partials + ([PARTIAL] if os.path.exists(PARTIAL) else []):
        with open(path, encoding="utf-8") as f:
            for line in f:
                fields = line.rstrip("\n").split("\t")
                if len(fields) != 4:  # a line cut by an upload taken mid-append
                    continue
                i, p, text, score = fields
                i = int(i)
                if i < len(rows) and rows[i][0] == p:  # same line of the same label files
                    preds[i] = (text, float(score))
    with open(PARTIAL, "w", encoding="utf-8") as f:  # everything done so far, so this session's output is complete
        f.writelines(f"{i}\t{rows[i][0]}\t{t}\t{s}\n" for i, (t, s) in sorted(preds.items()))
    print(len(preds), "lines predicted earlier")

    WORK = "/kaggle/working"
    if DEVICE == "gpu":
        # each process sees one GPU (CUDA_VISIBLE_DEVICES) as gpu:0
        workers = [(dict(model_dir=f"{MODEL_DIR}/inference", device="gpu:0"), {"CUDA_VISIBLE_DEVICES": str(k)})
                   for k in range(2)]
    else:
        workers = [(dict(model_dir=f"{MODEL_DIR}/onnx", device="cpu", engine="onnxruntime",
                         engine_config={"providers": ["OpenVINOExecutionProvider"], "provider_options": {"device_type": "CPU"}}),
                    {})]
    todo = [i for i in range(len(rows)) if i not in preds]
    for name in os.listdir(WORK):  # parts of an earlier run in this session are already in preds_partial.tsv
        if name.startswith("preds_part") and name != "preds_partial.tsv":
            os.remove(os.path.join(WORK, name))


    def part_lines(k):
        path = f"{WORK}/preds_part{k}.tsv"
        if not os.path.exists(path):
            return 0
        with open(path, encoding="utf-8") as f:
            return sum(1 for _ in f)


    uploader = None
    if PREDS_DATASET:
        try:
            from kaggle_secrets import UserSecretsClient
            os.environ["KAGGLE_API_TOKEN"] = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
            sh(f"{PIP} --upgrade kaggle")
            sys.path.insert(0, "ru_ocr/tools")
            from ckpt_uploader import CheckpointUploader
            uploader = CheckpointUploader("/kaggle/working", "/tmp/preds_upload", PREDS_DATASET, poll_s=600, settle_s=5,
                                          patterns=("preds_part*.tsv",), required="preds_part").start()
            print("preds_partial.tsv is uploaded to", PREDS_DATASET)
        except Exception as exc:
            print("predictions are NOT uploaded (no KAGGLE_API_TOKEN secret?):", exc)

    start = time.time()
    print(len(todo), "lines to predict", flush=True)
    procs = []
    try:
        for k, (kwargs, env) in enumerate(workers):
            with open(f"{WORK}/pred_todo{k}.tsv", "w", encoding="utf-8") as f:
                f.writelines(f"{i}\t{rows[i][0]}\t{os.path.join(ROOT, rows[i][0])}\n" for i in todo[k :: len(workers)])
            procs.append(subprocess.Popen(
                [sys.executable, f"{WORK}/pred_worker.py", f"{WORK}/pred_todo{k}.tsv", f"{WORK}/preds_part{k}.tsv",
                 json.dumps(kwargs), MODEL_NAME, str(BATCH), str(SAVE_EVERY)],
                stdout=open(f"{WORK}/pred_worker{k}.log", "w"), stderr=subprocess.STDOUT, env=dict(os.environ, **env)))
        while any(p.poll() is None for p in procs):
            time.sleep(60)
            done = sum(part_lines(k) for k in range(len(workers)))
            speed = done / (time.time() - start)
            eta = (len(todo) - done) / speed / 60 if speed else float("nan")
            print(f"{time.strftime('%H:%M:%S')} {done} / {len(todo)} lines, {speed:.0f} lines/s, ~{eta:.0f} min left", flush=True)
    finally:
        for proc in procs:
            if proc.poll() is None:
                proc.kill()
        if uploader is not None:
            uploader.stop()  # uploads the last predictions, also after an error
            print("uploads to", PREDS_DATASET, ":", uploader.uploads)
    codes = [proc.wait() for proc in procs]
    for k, code in enumerate(codes):
        if code:
            print(f"===== worker {k} exit code {code}, log tail\n", open(f"{WORK}/pred_worker{k}.log", errors="replace").read()[-3000:])
    assert not any(codes), "a worker failed, see its log above"
    for k in range(len(workers)):  # all parts into preds and preds_partial.tsv
        with open(f"{WORK}/preds_part{k}.tsv", encoding="utf-8") as f:
            for line in f:
                i, p, text, score = line.rstrip("\n").split("\t")
                preds[int(i)] = (text, float(score))
    with open(PARTIAL, "w", encoding="utf-8") as f:
        f.writelines(f"{i}\t{rows[i][0]}\t{t}\t{s}\n" for i, (t, s) in sorted(preds.items()))
    missing = sum(i not in preds for i in range(len(rows)))
    assert not missing, f"{missing} lines without a prediction"
    print(f"done in {(time.time() - start) / 60:.1f} min")
else:
    print("skipped: RUN_PREDICTIONS = False")

## 5. Disagreements

In [ ]:
if RUN_PREDICTIONS:  # off: export, check and test only
    import pandas as pd

    # preds_all.tsv.gz, review.tsv (differ in characters), review_spaces.tsv (only in spaces): ru_ocr/tools/build_review.py
    sh(f"{shlex.quote(sys.executable)} ru_ocr/tools/build_review.py --preds {PARTIAL} --labels "
       + " ".join(os.path.join(labels_dir, name) for name in LABEL_FILES) + " --out /kaggle/working")
    diff = pd.read_csv("/kaggle/working/review.tsv", sep="\t", quoting=3, keep_default_na=False)
    print(diff.groupby(pd.cut(diff.dist_ns, [0, 1, 2, 5, 10, 20, 1000]), observed=False).size().rename("lines by edits without spaces"))
    print(diff.groupby(pd.cut(diff.score, [0, 0.5, 0.8, 0.9, 0.95, 1.0]), observed=False).size().rename("lines by score"))
    pd.set_option("display.max_colwidth", 80)
    diff.head(30)[["label", "pred", "score", "dist_ns"]]
else:
    print("skipped: RUN_PREDICTIONS = False")